# MicroCLIP — Colab Evaluation

Zero-shot classification (CIFAR-10/100) and COCO 5K retrieval for a trained checkpoint.

Any GPU runtime works (eval is cheap — an A100 is overkill but fine).
Checkpoints are read from the Drive mirror written by the training notebook
(`MyDrive/microclip/runs/<run_name>/best.pt`) — run its final sync cell first.
Only `val2017` (~1 GB) and CIFAR go to local disk; Drive only gets small result JSONs.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

PERSIST = "/content/drive/MyDrive/microclip"

In [ ]:
%cd /content
!git clone https://github.com/umutonuryasar/microclip.git 2>/dev/null || git -C microclip pull
%cd /content/microclip
!pip -q install -e .

In [ ]:
import os

# Tokenizer written by the training notebook. Checkpoints are read straight from
# Drive paths below — no runs/ symlink, so nothing local ever gets rm -rf'd.
if not os.path.islink("artifacts"):
    os.system("rm -rf artifacts")
    os.symlink(f"{PERSIST}/artifacts", "artifacts")
!ls {PERSIST}/runs/

In [ ]:
%%bash
# Retrieval needs val2017 + annotations only.
mkdir -p data/coco && cd data/coco
if [ ! -d val2017 ]; then
  wget -q -c http://images.cocodataset.org/zips/val2017.zip
  unzip -q val2017.zip && rm val2017.zip
fi
if [ ! -d annotations ]; then
  wget -q -c http://images.cocodataset.org/annotations/annotations_trainval2017.zip
  unzip -q annotations_trainval2017.zip && rm annotations_trainval2017.zip
fi

## Evaluate

Set `CONFIG` to the config the checkpoint was trained with; the run folder is taken
from its `run_name` (e.g. `configs/ablations/optimizer_sgd.yml` → `runs/abl_sgd/`).
Results are saved under `Drive/microclip/results/` only if the eval succeeds.

In [ ]:
import json, os, shutil, torch
from microclip.config import load_config

CONFIG = "configs/sigmoid_b512.yml"

cfg = load_config(CONFIG)
RUN = cfg["run_name"]
REMOTE = f"{PERSIST}/runs/{RUN}"
CHECKPOINT = f"{REMOTE}/best.pt"
assert os.path.exists(CHECKPOINT), f"{CHECKPOINT} missing — wrong CONFIG, or training never synced"

# Warn if training did not finish (or its final sync was skipped).
steps = []
for slot in ("last_a.pt", "last_b.pt", "last.pt"):
    try:
        s = torch.load(f"{REMOTE}/{slot}", map_location="cpu", weights_only=False)
        steps.append((s["global_step"], s["epoch"]))
    except Exception:
        pass
if steps:
    step, epoch = max(steps)
    total = cfg["train"]["epochs"]
    print(f"{RUN}: newest Drive checkpoint at epoch {epoch}/{total} (step {step})")
    if epoch < total:
        print("WARNING: training incomplete on Drive — best.pt may not be the final best")

os.makedirs(f"{PERSIST}/results", exist_ok=True)


def run_eval(task: str) -> dict:
    local = f"/content/{RUN}_{task}.json"
    get_ipython().system(
        f"python scripts/evaluate.py --checkpoint '{CHECKPOINT}' --config {CONFIG} "
        f"--task {task} > {local}")
    with open(local) as f:
        results = json.load(f)  # raises if eval crashed -> previous Drive result untouched
    shutil.copyfile(local, f"{PERSIST}/results/{RUN}_{task}.json")
    print(json.dumps(results, indent=2))
    return results

In [ ]:
zeroshot = run_eval("zeroshot")

In [ ]:
retrieval = run_eval("retrieval")